## Example 1: asttokens functionality test 1

In [ ]:
import asttokens
import ast
import tokenize

# Example 1: Basic AST token annotation
print("Example 1: Basic AST token annotation")

# Sample Python code to analyze
source_code = """
def greet(name):
    message = f"Hello, {name}!"
    return message

result = greet("World")
print(result)
"""

# Create ASTTokens object
atok = asttokens.ASTTokens(source_code, parse=True)

print(f"Source code analyzed:")
print(f"  Lines: {source_code.count(chr(10)) + 1}")
print(f"  Characters: {len(source_code)}")

# Access the AST tree
ast_tree = atok.tree
print(f"\nAST information:")
print(f"  Root node type: {type(ast_tree).__name__}")
print(f"  AST nodes: {len(list(ast.walk(ast_tree)))}")

# Get token information
tokens = atok.tokens
print(f"  Token count: {len(tokens)}")
print(f"  First few tokens:")
for i, token in enumerate(tokens[:5]):
    token_name = tokenize.tok_name.get(token.type, f"UNKNOWN({token.type})")
    print(f"    {i}: {token_name} = '{token.string}'")

# The tokens cover the source exactly, and each node maps back to its own text
assert atok.text == source_code and type(ast_tree) is ast.Module
assert all(source_code[token.startpos:token.endpos] == token.string for token in tokens)
func_text = atok.get_text(ast_tree.body[0])
print(f"\nSource of greet(): {func_text!r}")
assert func_text == 'def greet(name):\n    message = f"Hello, {name}!"\n    return message', func_text

print("Basic AST token annotation completed")

## Example 2: asttokens functionality test 2

In [ ]:
import asttokens
import ast

# Example 2: Getting source text for AST nodes
print("Example 2: Getting source text for AST nodes")

# More complex code example
source_code = """
class Calculator:
    def add(self, x, y):
        return x + y
    
    def multiply(self, x, y):
        return x * y

calc = Calculator()
result1 = calc.add(5, 3)
result2 = calc.multiply(4, 7)
"""

atok = asttokens.ASTTokens(source_code, parse=True)

print(f"Analyzing complex source code...")

# Find different types of AST nodes
for node in ast.walk(atok.tree):
    if isinstance(node, ast.ClassDef):
        class_text = atok.get_text(node)
        print(f"\nClass definition found:")
        print(f"  Name: {node.name}")
        print(f"  Source length: {len(class_text)} characters")
        print(f"  First line: {class_text.split(chr(10))[0]}")
    
    elif isinstance(node, ast.FunctionDef):
        func_text = atok.get_text(node)
        print(f"\nFunction definition:")
        print(f"  Name: {node.name}")
        print(f"  Source: {func_text.strip()}")
        
    elif isinstance(node, ast.Call) and hasattr(node.func, 'attr'):
        call_text = atok.get_text(node)
        print(f"\nMethod call:")
        print(f"  Call: {call_text}")
        print(f"  Method: {node.func.attr}")

# Check the extracted text of every function and method call
function_texts = {node.name: atok.get_text(node).strip() for node in ast.walk(atok.tree) if isinstance(node, ast.FunctionDef)}
assert function_texts == {
    'add': 'def add(self, x, y):\n        return x + y',
    'multiply': 'def multiply(self, x, y):\n        return x * y'
}, function_texts
call_texts = sorted(atok.get_text(node) for node in ast.walk(atok.tree) if isinstance(node, ast.Call))
assert call_texts == ['Calculator()', 'calc.add(5, 3)', 'calc.multiply(4, 7)'], call_texts
class_node = next(node for node in ast.walk(atok.tree) if isinstance(node, ast.ClassDef))
start, end = atok.get_text_range(class_node)
assert source_code[start:end] == atok.get_text(class_node) and atok.get_text(class_node).startswith('class Calculator:')

print("\nGetting source text for AST nodes completed")

## Example 3: asttokens functionality test 3

In [ ]:
import asttokens
import ast

# Example 3: Token ranges and precise text extraction
print("Example 3: Token ranges and text extraction")

# Code with various expressions
source_code = """
data = [1, 2, 3, 4, 5]
result = sum(x * 2 for x in data if x > 2)
name = "Python"
formatted = f"Hello {name}!"
"""

atok = asttokens.ASTTokens(source_code, parse=True)

print(f"Analyzing expressions and their ranges...")

# Find different expression types
for node in ast.walk(atok.tree):
    if isinstance(node, ast.ListComp):
        list_comp_text = atok.get_text(node)
        print(f"\nList comprehension:")
        print(f"  Text: {list_comp_text}")
        print(f"  Start pos: {node.first_token.startpos}")
        print(f"  End pos: {node.last_token.endpos}")
        
    elif isinstance(node, ast.JoinedStr):  # f-string
        fstring_text = atok.get_text(node)
        print(f"\nF-string:")
        print(f"  Text: {fstring_text}")
        print(f"  Token range: {node.first_token.startpos}-{node.last_token.endpos}")
        
    elif isinstance(node, ast.List):
        list_text = atok.get_text(node) 
        print(f"\nList literal:")
        print(f"  Text: {list_text}")
        print(f"  Elements: {len(node.elts)}")

# Show token-to-position mapping
print(f"\nToken position mapping:")
sample_tokens = atok.tokens[:10]
for token in sample_tokens:
    if token.string.strip():  # Skip whitespace tokens
        print(f"  '{token.string}' at positions {token.startpos}-{token.endpos}")

# Check the text of each expression and that its token range points back into the source
texts = {}
for node in ast.walk(atok.tree):
    if isinstance(node, (ast.List, ast.JoinedStr, ast.GeneratorExp)):
        texts[type(node).__name__] = atok.get_text(node)
        assert source_code[node.first_token.startpos:node.last_token.endpos] == atok.get_text(node)
assert texts == {
    'List': '[1, 2, 3, 4, 5]',
    'GeneratorExp': '(x * 2 for x in data if x > 2)',
    'JoinedStr': 'f"Hello {name}!"'
}, texts

print("Token ranges and text extraction completed")

## Example 4: asttokens functionality test 4

In [ ]:
import asttokens
import ast

# Example 4: Working with existing AST trees
print("Example 4: Working with existing AST trees")

# Create AST tree first, then annotate with tokens
source_code = """
import math

def circle_area(radius):
    '''Calculate the area of a circle'''
    if radius < 0:
        raise ValueError("Radius cannot be negative")
    return math.pi * radius ** 2

areas = [circle_area(r) for r in [1, 2, 3, 4, 5]]
total = sum(areas)
"""

# Parse AST first
tree = ast.parse(source_code)

# Then create ASTTokens with existing tree
atok = asttokens.ASTTokens(source_code, tree=tree)

print(f"Working with pre-existing AST tree...")

# Analyze the AST structure
node_types = {}
for node in ast.walk(tree):
    node_type = type(node).__name__
    node_types[node_type] = node_types.get(node_type, 0) + 1

print(f"\nAST node type distribution:")
for node_type, count in sorted(node_types.items()):
    print(f"  {node_type}: {count}")

# Extract specific constructs
for node in ast.walk(tree):
    if isinstance(node, ast.FunctionDef):
        func_text = atok.get_text(node)
        docstring = ast.get_docstring(node)
        print(f"\nFunction '{node.name}':")
        print(f"  Arguments: {len(node.args.args)}")
        print(f"  Has docstring: {docstring is not None}")
        print(f"  Source length: {len(func_text)} characters")
        
    elif isinstance(node, ast.Import):
        import_text = atok.get_text(node)
        print(f"\nImport statement:")
        print(f"  Text: {import_text.strip()}")
        print(f"  Modules: {[alias.name for alias in node.names]}")

# The pre-parsed tree is the one asttokens annotated
assert atok.tree is tree
assert node_types['FunctionDef'] == 1 and node_types['Import'] == 1 and node_types['ListComp'] == 1, node_types
circle_area = next(node for node in ast.walk(tree) if isinstance(node, ast.FunctionDef))
assert ast.get_docstring(circle_area) == 'Calculate the area of a circle'
assert atok.get_text(circle_area).startswith('def circle_area(radius):')
assert atok.get_text(next(node for node in ast.walk(tree) if isinstance(node, ast.Import))) == 'import math'

print("Working with existing AST trees completed")

## Example 5: asttokens functionality test 5

In [ ]:
import asttokens
import ast
import tokenize

# Example 5: ASTTokens utilities and advanced features
print("Example 5: ASTTokens utilities and advanced features")

# Complex code example with multiple constructs
source_code = """
# Configuration and utilities
import sys
from collections import defaultdict

class DataProcessor:
    def __init__(self, name="default"):
        self.name = name
        self.data = defaultdict(list)
    
    def process(self, items):
        return [item.upper() if isinstance(item, str) else item * 2 
                for item in items if item is not None]

# Usage example
processor = DataProcessor("main")
sample_data = ["hello", None, 42, "world"]
processed = processor.process(sample_data)
print(f"Processed: {processed}")
"""

atok = asttokens.ASTTokens(source_code, parse=True)

print(f"ASTTokens package capabilities analysis:")

# Show package information
print(f"\nPackage features:")
atok_attrs = [attr for attr in dir(atok) if not attr.startswith('_')]
print(f"  Available methods: {len(atok_attrs)}")
print(f"  Key methods: get_text, get_text_range, tokens, tree")

# Detailed analysis of constructs
constructs_found = {
    'classes': 0,
    'methods': 0,  
    'imports': 0,
    'comprehensions': 0,
    'function_calls': 0
}

for node in ast.walk(atok.tree):
    if isinstance(node, ast.ClassDef):
        constructs_found['classes'] += 1
        print(f"\nClass: {node.name}")
        class_text = atok.get_text(node)
        print(f"  Length: {len(class_text)} characters")
        
    elif isinstance(node, ast.FunctionDef):
        constructs_found['methods'] += 1
        
    elif isinstance(node, (ast.Import, ast.ImportFrom)):
        constructs_found['imports'] += 1
        
    elif isinstance(node, (ast.ListComp, ast.DictComp)):
        constructs_found['comprehensions'] += 1
        
    elif isinstance(node, ast.Call):
        constructs_found['function_calls'] += 1

print(f"\nCode structure analysis:")
for construct, count in constructs_found.items():
    print(f"  {construct}: {count}")
assert constructs_found == {
    'classes': 1,
    'methods': 2,  # __init__ and process
    'imports': 2,
    'comprehensions': 1,
    'function_calls': 6  # defaultdict, isinstance, upper, DataProcessor, process, print
}, constructs_found

# Token statistics
token_types = {}
for token in atok.tokens:
    token_type = tokenize.tok_name.get(token.type, f"UNKNOWN({token.type})")
    token_types[token_type] = token_types.get(token_type, 0) + 1

print(f"\nToken type distribution:")
for token_type, count in sorted(token_types.items())[:5]:
    print(f"  {token_type}: {count}")
assert sum(token_types.values()) == len(atok.tokens) and token_types['COMMENT'] == 2, token_types

print(f"\nASTTokens capabilities demonstrated:")
print("- Basic AST token annotation")
print("- Source text extraction for nodes")  
print("- Token ranges and position mapping")
print("- Working with existing AST trees")
print("- Utilities and advanced analysis features")

print("ASTTokens utilities and advanced features completed")